In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from sklearn.cluster import KMeans

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.1)

# Set the plot style for consistency
SAVE_PATH = "plots/"
os.makedirs(SAVE_PATH, exist_ok=True)

## 1. Load data

In [2]:
# Load FPS results for GPU (3080 Ti) and TPU from CSVs
df_gpu_320 = pd.read_csv("fps_3080ti_320_results.csv")
df_gpu_320.yolo_model = df_gpu_320.yolo_model.str.replace('.pt', '')
df_gpu_512 = pd.read_csv("fps_3080ti_512_results.csv")
df_gpu_512.yolo_model = df_gpu_512.yolo_model.str.replace('.pt', '')
df_tpu_320 = pd.read_csv("fps_tpu_320_results.csv")
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("../tpu_weights/v8/320/", '')
df_tpu_512 = pd.read_csv("fps_tpu_512_results.csv")
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("../tpu_weights/v8/512/", '')

df_gpu_320["hardware"] = "GPU"
df_gpu_320["img_size"] = 320
df_gpu_512["hardware"] = "GPU"
df_gpu_512["img_size"] = 512
df_tpu_320["hardware"] = "TPU"
df_tpu_320["img_size"] = 320
df_tpu_512["hardware"] = "TPU"
df_tpu_512["img_size"] = 512

df_fps = pd.concat([df_gpu_320, df_gpu_512, df_tpu_320, df_tpu_512], ignore_index=True)
df_fps.drop_duplicates(inplace=True)
df_fps = df_fps.drop(columns=['avg_time_per_frame'])
print(len(df_fps))
print(df_fps.columns)
df_fps.sample(5)

3660
Index(['tracker', 'yolo_model', 'reid_model', 'object_count', 'avg_fps',
       'min_time_per_frame', 'max_time_per_frame', 'hardware', 'img_size'],
      dtype='object')


,tracker,yolo_model,reid_model,object_count,avg_fps,min_time_per_frame,max_time_per_frame,hardware,img_size
3491,botsort,yolov8s,osnet_ibn_x1_0_msmt17.pt,2,2.337526,416.5,771.5,TPU,512
743,botsort,yolov8s,clip_market1501.pt,4,42.987417,21.8,29.5,GPU,320
1421,strongsort,yolov8s,lmbn_n_market.pt,2,33.435729,21.7,61.0,GPU,512
2855,ocsort,yolov8s,osnet_ain_x1_0_msmt17.pt,1,23.317178,36.6,73.5,TPU,320
3049,deepocsort,yolov8n,clip_market1501.pt,5,0.467089,2047.4,2399.6,TPU,320


In [3]:
benchmarks = []
for name in os.listdir('.'):
    if "results_" not in name:
        continue
    fps = int(name.split('_')[1][:-3])
    df = pd.read_csv(f"{name}/results.csv")
    df['fps'] = fps
    benchmarks.append(df.copy())
df_benchmarks = pd.concat(benchmarks, ignore_index=True)
df_benchmarks["YOLO Model"] = df_benchmarks["YOLO Model"].str.split("_").str[0]
df_benchmarks = df_benchmarks.drop(columns=['FPS', 'Elapsed_time', 'Status'])

df_benchmarks.rename(columns={'Tracker': 'tracker', 'YOLO Model': 'yolo_model', "REID Model": "reid_model", "ImgSz": 'img_size', "HOTA": 'hota',  "MOTA": 'mota',  "IDF1": 'idf1'}, inplace=True)
df_benchmarks.drop_duplicates(inplace=True)
print(len(df_benchmarks))
print(df_benchmarks.columns)
df_benchmarks.sample(5)

2880
Index(['tracker', 'reid_model', 'yolo_model', 'img_size', 'hota', 'mota',
       'idf1', 'fps'],
      dtype='object')


,tracker,reid_model,yolo_model,img_size,hota,mota,idf1,fps
1499,strongsort,lmbn_n_market.pt,yolov8x,320,41.130,44.665,54.277,30
846,deepocsort,clip_market1501.pt,yolov8s,320,22.978,18.861,28.069,1
888,imprassoc,osnet_x1_0_market1501.pt,yolov8l,320,31.561,31.534,38.584,1
207,bytetrack,clip_market1501.pt,yolov8m,320,36.033,42.742,48.428,15
353,deepocsort,osnet_x0_25_market1501.pt,yolov8l,512,40.750,47.747,54.868,15


In [4]:
import pandas as pd

# Create a copy of df_fps and rename 'avg_fps' to 'fps_eval'
df_fps_copy = df_fps.copy().rename(columns={'avg_fps': 'fps_eval'})

# Add an index column to track each row
df_fps_copy = df_fps_copy.reset_index().rename(columns={'index': 'fps_index'})

# Merge with df_benchmarks on the common keys; include 'img_size' if needed
merged = pd.merge(
    df_fps_copy,
    df_benchmarks,
    on=['tracker', 'yolo_model', 'reid_model', 'img_size'],
    suffixes=('', '_bench')  # Only benchmark has 'fps'
)

# Compute the absolute difference between df_fps's fps_eval and benchmark's fps
merged['diff'] = (merged['fps_eval'] - merged['fps']).abs()

# For each original df_fps row (identified by fps_index), choose the benchmark row with the smallest fps difference
best_matches = merged.sort_values('diff').groupby('fps_index', as_index=False).first()

# Rename the benchmark's fps column to 'fps_benсh'
best_matches = best_matches.rename(columns={'fps': 'fps_benсh'})

# Merge back the selected benchmark columns ('hota', 'mota', 'idf1', and 'fps_benсh') to the original df_fps_copy
df_all = pd.merge(
    df_fps_copy,
    best_matches[['fps_index', 'fps_benсh', 'hota', 'mota', 'idf1']],
    on='fps_index',
    how='left'
)

# Optionally, drop the temporary 'fps_index' column
df_all = df_all.drop(columns='fps_index')

# Now df_all is a copy of df_fps (with 'fps_eval' instead of 'avg_fps') 
# and with added benchmark columns, where the benchmark's fps column is renamed to 'fps_benсh'
df_all[df_all.fps_eval <30].sample(5)

,tracker,yolo_model,reid_model,object_count,fps_eval,min_time_per_frame,max_time_per_frame,hardware,img_size,fps_benсh,hota,mota,idf1
3497,botsort,yolov8s,osnet_ain_x1_0_msmt17.pt,3,1.332425,713.7,907.5,TPU,512,1,27.861,27.293,34.679
3576,deepocsort,yolov8s,osnet_ain_x1_0_msmt17.pt,2,2.311745,405.8,895.5,TPU,512,3,35.037,36.269,46.292
3163,imprassoc,yolov8s,lmbn_n_market.pt,4,0.806073,1187.6,1648.1,TPU,320,1,30.529,30.547,37.240
2841,ocsort,yolov8s,lmbn_n_market.pt,2,24.144420,40.8,43.8,TPU,320,30,38.265,40.245,51.373
2929,bytetrack,yolov8s,clip_market1501.pt,5,27.323239,36.2,38.9,TPU,320,30,38.979,41.386,52.651
